In [1]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import types
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/11 12:04:02 WARN Utils: Your hostname, LAPTOP-NH6TPSRN, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/03/11 12:04:02 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/11 12:04:03 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
spark.version

'4.1.1'

In [38]:
df = spark.read \
    .parquet('data/raw/yellow/2025/11/*')

26/03/11 14:37:28 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: data/raw/yellow/2025/11/*.
java.io.FileNotFoundException: File data/raw/yellow/2025/11/* does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:970)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:384)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(ResolveDataSource.scala:143)
	at org.apache.spark.sql.catalyst.anal

In [39]:
df.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       7| 2025-11-01 00:13:25|  2025-11-01 00:13:25|              1|         1.68|         1|                 N|          43|    

In [49]:
df.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



In [ ]:
df \
     .repartition(4) \
     .write.parquet('06-batch/pq_yelloew_11_26')

In [42]:
df.registerTempTable('df')

In [43]:
df_result = spark.sql("""
SELECT to_date(tpep_pickup_datetime) as Date,
        count(1) as cnt
        
FROM df
WHERE to_date(tpep_pickup_datetime)= '2025-11-15'
GROUP BY to_date(tpep_pickup_datetime) 
    LIMIT 1
""")

df_result.show()

+----------+------+
|      Date|   cnt|
+----------+------+
|2025-11-15|162604|
+----------+------+



In [44]:
df_result = spark.sql("""
SELECT 
--    tpep_pickup_datetime, 
--    tpep_dropoff_datetime,
    max((unix_timestamp(tpep_dropoff_datetime) - unix_timestamp(tpep_pickup_datetime)) / 3600) as duration_hours
FROM df
LIMIT 1
""")

df_result.show()

+-----------------+
|   duration_hours|
+-----------------+
|90.64666666666666|
+-----------------+



In [45]:
zones = spark.read \
    .option("header", "true") \
    .csv('taxi_zone_lookup.csv')

In [48]:
zones.printSchema()

root
 |-- LocationID: string (nullable = true)
 |-- Borough: string (nullable = true)
 |-- Zone: string (nullable = true)
 |-- service_zone: string (nullable = true)



In [50]:
df_join = df.join(zones,df.PULocationID == zones.LocationID)

In [51]:
df_join.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)
 |-- LocationID: string (nullable = true)
 |-- Borough: string (nullable = t

In [55]:
df_join \
    .repartition(4) \
    .write.parquet('06-batch/reports/pq_yellow_11_26_with_zones', mode='overwrite')

In [57]:
df_join_z = spark.read.parquet('06-batch/reports/pq_yellow_11_26_with_zones/*')

26/03/11 14:56:47 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: 06-batch/reports/pq_yellow_11_26_with_zones/*.
java.io.FileNotFoundException: File 06-batch/reports/pq_yellow_11_26_with_zones/* does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:970)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:384)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(ResolveDataSource.scala:143

In [59]:
df_join_z.registerTempTable('df_join_z')

In [81]:
df_join_z = spark.sql("""
SELECT PULocationID  , Zone,
   count(*) as cnt_trips
FROM df_join_z
GROUP BY PULocationID , Borough , Zone
order by cnt_trips ,PULocationID  , Zone 
""")

df_join_z.show()

+------------+--------------------+---------+
|PULocationID|                Zone|cnt_trips|
+------------+--------------------+---------+
|           5|       Arden Heights|        1|
|          84|Eltingville/Annad...|        1|
|         105|Governor's Island...|        1|
|         187|       Port Richmond|        3|
|         109|         Great Kills|        4|
|         111| Green-Wood Cemetery|        4|
|         199|       Rikers Island|        4|
|         204|   Rossville/Woodrow|        4|
|           2|         Jamaica Bay|        5|
|         251|         Westerleigh|       12|
|          59|        Crotona Park|       14|
|         172|New Dorp/Midland ...|       14|
|         176|             Oakwood|       14|
|         245|       West Brighton|       14|
|         253|       Willets Point|       15|
|          27|Breezy Point/Fort...|       16|
|         206|Saint George/New ...|       17|
|          30|       Broad Channel|       18|
|         156|     Mariners Harbor